# Amazon Bedrock Guardrails와 AgentCore Memory로 대화 보호

## 개요

이 튜토리얼에서는 Amazon Bedrock Guardrails와 AgentCore Memory를 통합하여 안전한 대화형 Agent를 만드는 방법을 살펴봅니다. 상호작용 전반의 대화 컨텍스트를 유지하면서 민감한 콘텐츠를 필터링하는 Agent를 구축합니다.

### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                          |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | Guardrails / Memory 통합                                  |
| Agent 유형          | 보호된 Memory 지원 Agent                                 |
| Agentic Framework   | Strands Agents                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                      |
| 주요 기능        | Guardrails, Memory 통합, 콘텐츠 필터링                |
| 예제 난이도  | 중급                                                     |
| 사용 SDK            | Amazon Bedrock Python SDK and Bedrock Memory SDK                 |

### 학습 내용

이 튜토리얼에서는 다음 내용을 학습합니다.
1. Agent용 Memory 리소스를 생성하는 방법
2. 콘텐츠 필터링을 적용한 Amazon Bedrock Guardrails를 구현하는 방법
3. Guardrails와 Memory 기능을 결합하는 사용자 지정 Hook을 구축하는 방법
4. 안전한 대화 기록만 선택적으로 저장하는 방법
5. 다양한 유형의 콘텐츠로 안전한 Agent를 테스트하는 방법

### 아키텍처

이 예제는 안전한 대화를 위해 Guardrails와 Memory를 통합하는 방법을 보여 줍니다.

<div style="text-align:left">
    <img src="guardrails_memory_flow.png" width="90%"/>
</div>

## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* AgentCore Memory 및 Amazon Bedrock 액세스 권한이 구성된 AWS 자격 증명
* Amazon Bedrock 모델 액세스(Claude Haiku 4.5)
* Amazon Bedrock Memory SDK

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
# 가져오기
import os
import boto3
import uuid
import logging
from typing import Dict
from strands import Agent
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from botocore.exceptions import ClientError
from strands.hooks import (
    HookProvider,
    HookRegistry,
)
from strands.experimental.hooks import AfterModelInvocationEvent
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig

# 구성
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("secure-agent")
REGION = os.getenv("AWS_REGION", "us-west-2")  # Agent용 AWS 리전
bedrock_client = boto3.client("bedrock", region_name=REGION)
bedrock_runtime_client = boto3.client("bedrock-runtime", region_name=REGION)
memory_client = MemoryClient(region_name=REGION)

## 1. Amazon Bedrock Guardrails 생성

이 섹션에서는 Agent의 콘텐츠 안전 정책을 적용하는 Guardrail을 생성합니다. Guardrails는 사용자 입력과 모델 출력 모두에 적용할 수 있는 안전 필터 역할을 합니다. 이 예제에서는 두 가지 정책으로 Guardrail을 구성합니다.

1. **입력 필터링**: 사용자의 모욕적 표현 차단
2. **출력 필터링**: 모델이 정치 주제를 다루지 못하도록 차단

이 접근 방식은 Guardrails가 대화의 양방향에서 서로 다른 유형의 문제가 있는 콘텐츠를 방지하는 방법을 보여 줍니다. 입력 필터는 서로 존중하는 대화 환경을 유지하며, 출력 필터는 모델이 잠재적으로 민감한 주제를 다루지 않도록 합니다.

최종 목표는 원하지 않는 메시지가 Memory에 저장되는 것을 막고 향후 컨텍스트에 적절한 콘텐츠만 저장되도록 하는 것입니다.

In [ ]:
# 이 요청의 고유 식별자
unique_id = str(uuid.uuid4())[:6]

# Guardrail 구성 정의
guardrail_name = f"SecureConversationGuardrail_{unique_id}"
guardrail_description = "Blocks insults in input and political content in output"

try:
    # Guardrail 생성
    response = bedrock_client.create_guardrail(
        name=guardrail_name,
        description=guardrail_description,
        # 입력에서 모욕적 표현 차단
        contentPolicyConfig={
            "filtersConfig": [
                {
                    "type": "INSULTS",
                    "inputStrength": "MEDIUM",
                    "outputStrength": "MEDIUM",
                    "inputModalities": ["TEXT"],
                    "outputModalities": ["TEXT"],
                    "inputAction": "BLOCK",
                    "outputAction": "NONE",
                    "inputEnabled": True,
                    "outputEnabled": False,
                }
            ],
            "tierConfig": {"tierName": "CLASSIC"},
        },
        # 출력에서 정치적 콘텐츠 차단
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "Politics",
                    "definition": "Content related to political leaders, elections, political parties, or government affairs",
                    "examples": [
                        "Who is the current president?",
                        "Tell me about the upcoming election",
                        "Explain the political situation in Congress",
                    ],
                    "type": "DENY",
                    "inputAction": "NONE",
                    "outputAction": "BLOCK",
                    "inputEnabled": False,
                    "outputEnabled": True,
                }
            ],
            "tierConfig": {"tierName": "CLASSIC"},
        },
        blockedInputMessaging="I'm sorry, but your message contains inappropriate language. Please rephrase your question without insults.",
        blockedOutputsMessaging="I apologize, but I cannot provide information on political topics. Is there something else I can help you with?",
    )

    # 나중에 사용할 Guardrail ID 저장
    guardrail_id = response["guardrailId"]
    guardrail_arn = response["guardrailArn"]
    guardrail_version = "DRAFT"  # 새 Guardrail은 DRAFT로 생성됨

    print(f"✅ Created guardrail: {guardrail_id} (ARN: {guardrail_arn})")

except Exception as e:
    print(f"❌ Error creating guardrail: {e}")
    # Guardrail이 이미 존재하면 ID 검색
    try:
        response = bedrock_client.list_guardrails()
        existing_guardrail = next(
            (g for g in response["guardrailSummaries"] if g["name"] == guardrail_name),
            None,
        )
        if existing_guardrail:
            guardrail_id = existing_guardrail["guardrailId"]
            guardrail_version = "DRAFT"  # DRAFT 버전 사용
            print(f"Using existing guardrail: {guardrail_id}")
    except Exception as list_error:
        print(f"❌ Error listing guardrails: {list_error}")
        guardrail_id = None
        guardrail_version = None

## 2. Memory 리소스 생성

이 섹션에서는 Agent가 대화 기록을 저장할 Memory 리소스를 생성합니다. Memory를 사용하면 Agent가 과거 상호작용을 기억하고 컨텍스트를 유지하여 시간이 지나도 더 일관된 응답을 제공할 수 있습니다. Memory와 Guardrails를 결합하면 나중에 참조할 적절한 콘텐츠만 저장되도록 할 수 있습니다.

이 예제에서는 추가 strategy 없이 간단한 단기 Memory 리소스를 생성합니다. 이는 세션 내 대화 컨텍스트를 유지하는 데 적합합니다. Memory에는 Guardrail 검사를 통과한 메시지만 저장되므로 부적절한 콘텐츠가 필터링됩니다.

In [ ]:
memory_name = f"SecureAgentMemory_{unique_id}"

try:
    # strategy 없이 Memory 리소스 생성(단기 메모리만 사용)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 strategy를 사용하지 않음
        description="Short-term memory for personal agent with guardrails",
        event_expiry_days=7,
    )
    memory_id = memory["id"]
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = memory_client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 표시
    logger.error(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if "memory_id" in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

## 3. Bedrock Guardrails, Strands 및 AgentCore Memory 통합

이 섹션에서는 Guardrails와 Memory 기능을 통합하는 사용자 지정 Hook을 생성합니다. 구현 내용은 다음과 같습니다.

1. Amazon Bedrock Guardrails로 사용자 입력과 모델 출력 모두 검사
2. 부적절한 콘텐츠를 안전한 대체 콘텐츠로 교체
3. Guardrail 검사를 통과한 메시지만 Memory에 저장
4. Agent 초기화 시 Memory에서 과거 대화 컨텍스트 검색

이 접근 방식을 사용하면 Agent가 Memory 기능을 활용하면서도 안전한 대화 기록을 유지할 수 있습니다. 필요한 구성 요소를 구축해 보겠습니다.

In [ ]:
class GuardrailsEvaluator:
    """재사용 가능한 가드레일 평가 유틸리티입니다."""

    def __init__(self, guardrail_id: str, guardrail_version: str):
        """가드레일 평가기를 초기화합니다.

        인자:
            guardrail_id: 사용할 가드레일 ID
            guardrail_version: 가드레일 버전(예: "DRAFT")
        """
        self.guardrail_id = guardrail_id
        self.guardrail_version = guardrail_version

    def evaluate_content(self, content: str, source: str) -> Dict:
        """Bedrock Guardrails로 콘텐츠를 평가하고 결과를 반환합니다.

        인자:
            content: 평가할 텍스트 콘텐츠
            source: 소스 유형("INPUT" 또는 "OUTPUT")

        반환값:
            가드레일 평가 결과가 포함된 딕셔너리
        """
        try:
            logger.info(f"⏳ CHECKING {source}: '{content[:30]}...'")

            response = bedrock_runtime_client.apply_guardrail(
                guardrailIdentifier=self.guardrail_id,
                guardrailVersion=self.guardrail_version,
                source=source,
                content=[{"text": {"text": content}}],
            )

            action = response.get("action")
            logger.info(f"🔍 GUARDRAIL ACTION: {action}")

            return response
        except Exception as e:
            logger.error(f"❌ Guardrail evaluation failed: {e}")
            return {"error": str(e)}


class GuardrailsHookProvider(HookProvider):
    """가드레일 적용과 메모리 저장을 결합하는 훅 제공자입니다."""

    def __init__(self, guardrails_evaluator: GuardrailsEvaluator):
        self.evaluator = guardrails_evaluator
        self.blocked_outputs = set()

    def after_model_invocation(self, event: AfterModelInvocationEvent) -> None:
        """가드레일로 모델 출력을 확인하고 필요하면 교체합니다.

        인자:
            event: 모델 응답이 포함된 이벤트
        """
        # 모델 호출이 실패하면 건너뛰기
        if event.exception is not None or event.stop_response is None:
            logger.error("⚠️ Model invocation failed, skipping guardrail check")
            return

        logger.info("🔍 AfterModelInvocationEvent: Checking model output")

        # 모델 응답에서 메시지 추출
        message = event.stop_response.message

        # 콘텐츠 추출
        if isinstance(message.get("content"), list):
            content = "".join(block.get("text", "") for block in message.get("content", []))
        else:
            content = str(message.get("content", ""))

        content_id = hash(content)

        # Guardrails로 검사
        result = self.evaluator.evaluate_content(content, "OUTPUT")

        # Guardrail 위반 처리
        if result.get("action") == "GUARDRAIL_INTERVENED":
            logger.warning("⛔ ASSISTANT MESSAGE BLOCKED BY GUARDRAILS")

            # 이 출력을 차단됨으로 표시
            self.blocked_outputs.add(content_id)

            # Guardrail에서 제공한 대체 콘텐츠가 있으면 가져오기
            replacement_content = None
            if "outputs" in result and result["outputs"] and len(result["outputs"]) > 0:
                if "text" in result["outputs"][0]:
                    replacement_content = result["outputs"][0]["text"]

            # 대체 콘텐츠가 없으면 일반 메시지 사용
            if not replacement_content:
                replacement_content = "I apologize, but I cannot provide the requested information as it would violate our content policies."

            # 메시지 내용 업데이트 - 사용자가 보는 내용이 변경됨
            if isinstance(message.get("content"), list):
                message["content"] = [{"text": replacement_content}]
            else:
                message["content"] = replacement_content

            logger.info(f"⚠️ Replaced assistant message with guardrail response: {replacement_content[:30]}...")

    def register_hooks(self, registry: HookRegistry):
        """모든 훅을 레지스트리에 등록합니다.

        인자:
            registry: 훅을 등록할 레지스트리
        """
        registry.add_callback(AfterModelInvocationEvent, self.after_model_invocation)

## 4. Agent 생성 및 구성

이 섹션에서는 지금까지 구축한 Bedrock 모델, Guardrails Evaluator, Memory 지원 Hook Provider를 결합하여 안전한 대화형 Agent를 생성합니다. 이 통합으로 콘텐츠 정책을 적용하고 적절한 컨텍스트를 저장하면서 대화를 유지할 수 있는 완전한 Agent가 만들어집니다.

In [ ]:
ACTOR_ID = "user_1"
SESSION_ID = "session_001"
# bedrock_model = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0")

evaluator = GuardrailsEvaluator(guardrail_id=guardrail_id, guardrail_version=guardrail_version)

session_manager = None


def create_personal_agent():
    """메모리와 가드레일 기능이 있는 개인 에이전트를 생성합니다."""
    global session_manager
    # 이전 Session Manager가 있으면 종료
    if session_manager is not None:
        session_manager.close()

    # AgentCore Memory 구성
    config = AgentCoreMemoryConfig(memory_id=memory_id, session_id=SESSION_ID, actor_id=ACTOR_ID)

    # Session Manager 생성 - 명시적 수명 주기로 리소스 정리 셀에서 종료
    session_manager = AgentCoreMemorySessionManager(agentcore_memory_config=config, region_name=REGION)

    # Session Manager 및 Guardrails Hook으로 Agent 생성
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
        system_prompt="You are a helpful personal assistant. Be friendly and professional.",
        session_manager=session_manager,
        hooks=[GuardrailsHookProvider(evaluator)],
        callback_handler=None,
    )
    return agent


# Agent 생성
agent = create_personal_agent()
logger.info("✅ Personal agent created with memory and guardrails")

이 구현에서는 다음 기능을 갖춘 안전한 Agent를 생성합니다.

1. 초기화 시 Memory에서 기존 대화 컨텍스트 불러오기
2. 처리 전에 Guardrails로 사용자 입력 검사
3. 사용자에게 표시하기 전에 Guardrails로 모델 출력 검사
4. 향후 컨텍스트를 위해 승인된 메시지만 Memory에 저장
5. 여러 상호작용에서 대화 기록 유지

Guardrails와 Memory를 결합하면 Agent가 안전하면서도 컨텍스트에 맞는 대화 환경을 유지할 수 있습니다.

## 5. 안전한 Agent 테스트

다양한 유형의 입력으로 Agent를 테스트하여 Guardrails와 Memory 통합이 실제로 어떻게 작동하는지 살펴보겠습니다. 허용되는 입력과 Guardrail 개입을 trigger할 수 있는 입력을 모두 사용하여 구현이 올바르게 작동하는지 검증합니다.

먼저 Guardrail 검사와 Agent 호출을 처리하는 helper 함수를 생성합니다.

In [ ]:
def process_with_guardrails(user_input):
    """사용자 입력을 에이전트에 보내기 전에 가드레일로 처리합니다.

    인자:
        user_input: 사용자의 텍스트 입력

    반환값:
        에이전트 응답 또는 가드레일 거부 결과
    """
    # Guardrails로 입력 검사
    result = evaluator.evaluate_content(user_input, "INPUT")

    if result.get("action") == "GUARDRAIL_INTERVENED":
        # Guardrail의 거부 메시지 가져오기
        if "outputs" in result and result["outputs"] and "text" in result["outputs"][0]:
            rejection_content = result["outputs"][0]["text"]
        else:
            rejection_content = "I cannot process that request."

        # Agent를 호출하지 않고 거부 응답 반환
        print(rejection_content)
        return rejection_content
    else:
        # 입력이 Guardrail을 통과했으므로 Agent 호출 진행
        response = agent(user_input)
        print(response)
        return response

### 테스트 1: 일반 대화

모든 Guardrail을 통과해야 하는 일반적인 인사로 시작합니다.

In [ ]:
print("Test 1: Normal greeting")
user_input = "I am dani."
process_with_guardrails(user_input)

### 테스트 2: 모욕적 콘텐츠(입력 Guardrail이 개입해야 함)

입력 Guardrail에서 차단해야 하는 모욕적 표현을 입력해 봅니다.

In [ ]:
print("\nTest 2: Insulting content (should trigger input guardrail)")
user_input = "You're a stupid assistant."
process_with_guardrails(user_input)

### 테스트 3: 정치적 콘텐츠(출력 Guardrail이 개입해야 함)

이번에는 입력 Guardrail은 통과하지만 출력 Guardrail이 개입해야 하는 정치 관련 질문을 사용해 봅니다.

In [ ]:
print("\nTest 3: Political question (should trigger output guardrail)")
user_input = "Who is the president of the US?"
process_with_guardrails(user_input)

### Memory 내용 살펴보기

테스트 후 Memory에 저장된 내용을 확인해 보겠습니다.

In [ ]:
# Memory에 저장된 내용 확인
print("\n=== Memory Contents ===")
manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
session = manager.create_memory_session(actor_id=ACTOR_ID, session_id=SESSION_ID)
recent_turns = session.get_last_k_turns(k=5)


for i, turn in enumerate(recent_turns):
    print(f"\nTurn {i + 1}:")
    for msg in turn:
        role = msg["role"]
        content = msg["content"]["text"]
        print(f"- {role}: {content[:100]}...")

### 테스트 4: Memory 테스트를 위한 후속 질문
Agent가 이전 컨텍스트를 기억하는지 확인하기 위해 후속 질문을 해 보겠습니다.

In [ ]:
agent = create_personal_agent()
print("\nTest 4: Follow-up to test memory")
user_input = "What's my name?"
process_with_guardrails(user_input)

## 6. 리소스 정리(선택 사항)

안전한 Agent 실습을 마쳤다면 이 튜토리얼에서 생성한 리소스를 정리할 수 있습니다. 이 섹션에서는 Guardrail과 Memory 리소스를 삭제하는 방법을 살펴봅니다.

In [ ]:
# buffered 메시지를 flush하도록 Session Manager 종료
if session_manager is not None:
    session_manager.close()
    print("✅ Closed session manager")

# Memory 리소스 삭제
try:
    memory_client.delete_memory_and_wait(memory_id=memory_id)
    print(f"✅ Deleted memory resource: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")

# Guardrail 삭제
try:
    bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)
    print(f"✅ Deleted guardrail: {guardrail_id}")
except Exception as e:
    print(f"❌ Error deleting guardrail: {e}")

## 마무리

이 튜토리얼에서는 Amazon Bedrock Guardrails와 AgentCore Memory 기능을 결합한 안전한 대화형 Agent를 구축했습니다. 구현 내용은 다음과 같습니다.

1. Guardrails를 사용하여 부적절한 사용자 입력 필터링
2. Agent가 민감한 주제를 다루지 못하도록 차단
3. 승인된 메시지만 Memory에 저장
4. 향상된 사용자 경험을 위해 Memory로 대화 컨텍스트 유지

Guardrails와 Memory를 통합하면 콘텐츠 정책을 준수하면서도 개인화되고 컨텍스트에 맞는 응답을 제공하는 강력한 Agent를 구축할 수 있습니다. Guardrail 필터를 추가하거나 더 많은 장기 메모리 strategy를 구현하여 이 pattern을 더 복잡한 시나리오로 확장할 수 있습니다.